After noticing that warmup wasn't the source of difference, it seemed that two measurements with the same parameters
produced wastly different data. I started suspecting the data processing. I still don't have intuition for CV and it's
not a good metric for dispersion as it assumes normal distribution. I decided to plot histograms to see what is really
going on.

In [2]:
%use kandy
%use dataframe

In [3]:
@file:DependsOn("../build/libs/processing.jar")
import profilelib.*

In [15]:
typealias Data =  Map<Map<String, String>, Map<String, Int>>

In [5]:
import jdk.jfr.consumer.RecordedFrame

fun load_data(dataPath: String): Data =
    walkPath(dataPath)
        .filter { it.endsWith(".jfr") }
        .map { path ->
            val params = path
                .removeSuffix(".jfr")
                .substringAfterLast("/")
                .split(",")
                .map { it.split("=").let { (k, v) -> k to v} }
                .toMap()
            val freq = mutableMapOf<String, Int>()
            println("'" + path + "'")
            lazy_samples(path).toFreq(freq, when (params["platform"]) {
                "jvm" -> ::jvm_get_name
                "native" -> fun(frame: RecordedFrame): String = frame.method.name
                    .let { if (it.startsWith("kfun:"))
                        it.removePrefix("kfun:").replace("#", ".").substringBefore("(")
                    else
                        "weirdo:" + it
                    }
                else -> throw NotImplementedError("Unknown platform: ${params["platform"]}")
            })
            params to freq.toMap()
        }
        .toMap()

In [4]:
val base = load_data("../../jfrStorage/serialization-twitterBM/sampleInterval_and_cycles_variance_measurement")
val new_base = load_data("../../jfrStorage/serialization-twitterBM/new_base_with_memory")
val warmup = load_data("../../jfrStorage/serialization-twitterBM/warmup")

'../../jfrStorage/serialization-twitterBM/sampleInterval_and_cycles_variance_measurement/platform=jvm,cycles=5000,sampleInterval=5,iteration=22.jfr'
'../../jfrStorage/serialization-twitterBM/sampleInterval_and_cycles_variance_measurement/platform=jvm,cycles=50000,sampleInterval=100,iteration=109.jfr'
'../../jfrStorage/serialization-twitterBM/sampleInterval_and_cycles_variance_measurement/platform=native,cycles=100000,sampleInterval=5,iteration=9.jfr'
'../../jfrStorage/serialization-twitterBM/sampleInterval_and_cycles_variance_measurement/platform=native,cycles=50000,sampleInterval=1,iteration=20.jfr'
'../../jfrStorage/serialization-twitterBM/sampleInterval_and_cycles_variance_measurement/platform=jvm,cycles=5000,sampleInterval=100,iteration=34.jfr'
'../../jfrStorage/serialization-twitterBM/sampleInterval_and_cycles_variance_measurement/platform=jvm,cycles=100000,sampleInterval=100,iteration=46.jfr'
'../../jfrStorage/serialization-twitterBM/sampleInterval_and_cycles_variance_measurement

In [10]:
fun extract_data(data: Data, f: String): List<Int> = data
    .filter { it.key["platform"]!! == "jvm" && it.key["sampleInterval"]!! == "100" && it.key["cycles"]!! == "100000" && it.key["warmup"]?.let { it == "3"} ?: true }
    .map { it.value[f]!! }
    .toList()

In [8]:
import org.jetbrains.kotlinx.kandy.ir.Plot

fun plot_histogram(data: Map<String, List<Int>>): Plot {
    val list = data.toList()
    val df = dataFrameOf(
        "sample" to list.flatMap { it.second },
        "group" to list.flatMap { group -> group.second.map { group.first } }
    )

    return df.groupBy("group").plot {
        histogram("sample")
    }
}

In [25]:
val all_data = mapOf(
    "base" to base,
    "new_base" to new_base,
    "warmup" to warmup,
)

In [26]:
plot_histogram(all_data.mapValues { extract_data(it.value, "java.lang.String.substring") })

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="miSNwK"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"sample",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count",
"fill":"group",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup"],
"x":[85.74999999999999,102.89999999999998,120.04999999999998,137.2,154.34999999999997,171.5,188.64999999999998,205.79999999999995,222.95,240.09999999999997,257.24999999999994,274.4,291.54999999999995,308.7,325.84999999999997,342.99999999999994,360.1499999999999,377.29999999999995,394.45,411.59999999999997,85.74999999999999,102.89999999999998,120.04999999999998,137.2,154.34999999999997,171.5,188.64999999999998,205.79999999999995,222.95,240.09999999999997,257.24999999999994,274.4,291.54999999999995,308.7,325.84999999999997,342.99999999999994,360.1499999999999,377.29999999999995,394.45,411.59999999999997,85.74999999999999,102.89999999999998,120.04999999999998,137.2,154.34999999999997,171.5,188.64999999999998,205.79999999999995,222.95,240.09999999999997,257.24999999999994,274.4,291.54999999999995,308.7,325.84999999999997,342.99999999999994,360.1499999999999,377.29999999999995,394.45,411.59999999999997],
"count":[2.0,2.0,0.0,9.0,20.0,41.0,25.0,18.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,9.0,12.0,25.0,33.0,36.0,17.0,9.0,8.0,12.0,16.0,29.0,9.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,8.0,22.0,34.0,12.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
"group":["base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"group"
},{
"type":"float",
"column":"x"
},{
"type":"int",
"column":"count"
},{
"type":"str",
"column":"&merged_groups"
}]
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"group"
},{
"type":"str",
"column":"&merged_groups"
}]
},
"spec_id":"8"
};
 var containerDiv = document.getElementById("miSNwK");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 


In [27]:
plot_histogram(all_data.mapValues { extract_data(it.value, "org.jetbrains.stdlibprofiling.TwitterBenchmark.encodeDecode") })


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="XKav2S"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"sample",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count",
"fill":"group",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup"],
"x":[3728.874999999999,3793.7249999999995,3858.5749999999994,3923.4249999999993,3988.2749999999996,4053.124999999999,4117.974999999999,4182.824999999999,4247.674999999999,4312.525,4377.374999999999,4442.224999999999,4507.074999999999,4571.924999999999,4636.775,4701.624999999999,4766.474999999999,4831.324999999999,4896.174999999999,4961.025,3728.874999999999,3793.7249999999995,3858.5749999999994,3923.4249999999993,3988.2749999999996,4053.124999999999,4117.974999999999,4182.824999999999,4247.674999999999,4312.525,4377.374999999999,4442.224999999999,4507.074999999999,4571.924999999999,4636.775,4701.624999999999,4766.474999999999,4831.324999999999,4896.174999999999,4961.025,3728.874999999999,3793.7249999999995,3858.5749999999994,3923.4249999999993,3988.2749999999996,4053.124999999999,4117.974999999999,4182.824999999999,4247.674999999999,4312.525,4377.374999999999,4442.224999999999,4507.074999999999,4571.924999999999,4636.775,4701.624999999999,4766.474999999999,4831.324999999999,4896.174999999999,4961.025],
"count":[2.0,3.0,6.0,19.0,15.0,12.0,13.0,8.0,7.0,9.0,9.0,6.0,6.0,4.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,2.0,11.0,18.0,27.0,31.0,33.0,24.0,21.0,19.0,13.0,14.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,8.0,16.0,17.0,16.0,12.0,4.0,1.0,1.0,0.0,0.0,1.0],
"group":["base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","new_base","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup","warmup"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"group"
},{
"type":"float",
"column":"x"
},{
"type":"int",
"column":"count"
},{
"type":"str",
"column":"&merged_groups"
}]
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"group"
},{
"type":"str",
"column":"&merged_groups"
}]
},
"spec_id":"11"
};
 var containerDiv = document.getElementById("XKav2S");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 

So, it seems, that the results are really different. `base` and `warmup` should be the same but the distribution is
clearly different. Difference of `new_base` from the others could be explained by different `-Xms` and `-Xmx` settings
but I doubt that will have major effect. It seems there is a factor which influences the results that I don't have under
control.

I suspect the way I invoke the runs could play an effect. IIRC when I was measuring the `base` I first measured the K/N
and then K/JVM. This may not be clear from the code history as these measurements were not done by the *exact* code that
is in the repo but was reconstructed later (naively believing that the invocation shouldn't have na effect).

Therefore, more measurements should be done. Running two sets - one AAABBB and the second ABABAB and observing the
difference. Later, the invocation should be made robust such that this doesn't play an effect.

Also, measuring the effect of `-Xmx` or `-Xms` could be benefitial.

Also, trying a different machine may be a good idea. Currently, each measurement is done in a different time, power
supply (110V vs 230V), temperature conditions and possibly different workload of the computer (although I try to keep
that stable by switching off all other apps and running the script from CLI).

Furthermore, now, that I have better insight into the data, it also seems, that the variance between runs is much bigger
than expected. Therefore, investigating even higher number of samples seems necessary.

In [6]:
val aabb = load_data("../../jfrStorage/serialization-twitterBM/aabb_vs_abab/aabb")
val abab = load_data("../../jfrStorage/serialization-twitterBM/aabb_vs_abab/abab")

'../../jfrStorage/serialization-twitterBM/aabb_vs_abab/aabb/platform=jvm,cycles=100000,sampleInterval=100,iteration=43,warmup=3.jfr'
'../../jfrStorage/serialization-twitterBM/aabb_vs_abab/aabb/platform=native,cycles=100000,sampleInterval=100,iteration=10.jfr'
'../../jfrStorage/serialization-twitterBM/aabb_vs_abab/aabb/platform=jvm,cycles=100000,sampleInterval=100,iteration=44,warmup=3.jfr'
'../../jfrStorage/serialization-twitterBM/aabb_vs_abab/aabb/platform=jvm,cycles=100000,sampleInterval=100,iteration=50,warmup=3.jfr'
'../../jfrStorage/serialization-twitterBM/aabb_vs_abab/aabb/platform=jvm,cycles=100000,sampleInterval=100,iteration=20,warmup=3.jfr'
'../../jfrStorage/serialization-twitterBM/aabb_vs_abab/aabb/platform=jvm,cycles=100000,sampleInterval=100,iteration=48,warmup=3.jfr'
'../../jfrStorage/serialization-twitterBM/aabb_vs_abab/aabb/platform=jvm,cycles=100000,sampleInterval=100,iteration=67,warmup=3.jfr'
'../../jfrStorage/serialization-twitterBM/aabb_vs_abab/aabb/platform=native

In [7]:
val aabb_vs_abab = mapOf(
    "aabb" to aabb,
    "abab" to abab,
)

In [11]:
plot_histogram(aabb_vs_abab.mapValues { extract_data(it.value, "java.lang.String.substring") })

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="g1OEDM"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"sample",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count",
"fill":"group",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab"],
"x":[162.8,167.20000000000005,171.60000000000002,176.0,180.40000000000003,184.8,189.20000000000005,193.60000000000002,198.0,202.40000000000003,206.8,211.20000000000005,215.60000000000002,220.00000000000003,224.40000000000003,228.8,233.20000000000005,237.60000000000002,242.00000000000003,246.40000000000003,162.8,167.20000000000005,171.60000000000002,176.0,180.40000000000003,184.8,189.20000000000005,193.60000000000002,198.0,202.40000000000003,206.8,211.20000000000005,215.60000000000002,220.00000000000003,224.40000000000003,228.8,233.20000000000005,237.60000000000002,242.00000000000003,246.40000000000003],
"count":[0.0,0.0,1.0,4.0,7.0,7.0,9.0,4.0,13.0,5.0,12.0,11.0,8.0,9.0,3.0,4.0,1.0,1.0,1.0,0.0,2.0,2.0,3.0,2.0,3.0,4.0,2.0,13.0,17.0,8.0,12.0,12.0,8.0,5.0,2.0,3.0,0.0,0.0,1.0,1.0],
"group":["aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"group"
},{
"type":"float",
"column":"x"
},{
"type":"int",
"column":"count"
},{
"type":"str",
"column":"&merged_groups"
}]
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"group"
},{
"type":"str",
"column":"&merged_groups"
}]
},
"spec_id":"2"
};
 var containerDiv = document.getElementById("g1OEDM");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 160 
 
 
 
 
 
 
 
 
 180 
 
 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 220 
 
 
 
 
 
 
 
 
 240 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 12 
 
 
 
 
 
 
 14 
 
 
 
 
 
 
 16 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 sample 
 
 
 
 
 
 
 
 
 group 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 aabb 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abab

In [13]:
plot_histogram(aabb_vs_abab.mapValues { extract_data(it.value, "org.jetbrains.stdlibprofiling.TwitterBenchmark.encodeDecode") })

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="liOJ9a"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"sample",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count",
"fill":"group",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab"],
"x":[4018.95,4054.05,4089.1499999999996,4124.25,4159.35,4194.45,4229.55,4264.65,4299.75,4334.85,4369.95,4405.05,4440.15,4475.25,4510.35,4545.45,4580.55,4615.65,4650.75,4685.85,4018.95,4054.05,4089.1499999999996,4124.25,4159.35,4194.45,4229.55,4264.65,4299.75,4334.85,4369.95,4405.05,4440.15,4475.25,4510.35,4545.45,4580.55,4615.65,4650.75,4685.85],
"count":[1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,5.0,7.0,10.0,16.0,18.0,22.0,11.0,5.0,0.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,4.0,9.0,7.0,6.0,8.0,11.0,11.0,9.0,16.0,11.0,3.0,2.0,1.0,0.0,0.0],
"group":["aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","aabb","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab","abab"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"group"
},{
"type":"float",
"column":"x"
},{
"type":"int",
"column":"count"
},{
"type":"str",
"column":"&merged_groups"
}]
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"group"
},{
"type":"str",
"column":"&merged_groups"
}]
},
"spec_id":"5"
};
 var containerDiv = document.getElementById("liOJ9a");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 4,000 
 
 
 
 
 
 
 
 
 4,100 
 
 
 
 
 
 
 
 
 4,200 
 
 
 
 
 
 
 
 
 4,300 
 
 
 
 
 
 
 
 
 4,400 
 
 
 
 
 
 
 
 
 4,500 
 
 
 
 
 
 
 
 
 4,600 
 
 
 
 
 
 
 
 
 4,700 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 sample 
 
 
 
 
 
 
 
 
 group 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 aabb 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abab

In [20]:
fun do_statistics(data: List<Int>) {
    val mean = data.sum().toDouble() / data.size
    val variance = data.sumOf { (it.toDouble() - mean).let { it * it } } / data.size
    val stddev = Math.sqrt(variance)
    val cv = stddev / mean
    println("%.3f (%.3f)   ".format(mean, cv))
}

In [22]:
do_statistics(extract_data(aabb, "java.lang.String.substring"))
do_statistics(extract_data(abab, "java.lang.String.substring"))


203.220 (0.077)   
201.450 (0.078)   


In [23]:
do_statistics(extract_data(aabb, "org.jetbrains.stdlibprofiling.TwitterBenchmark.encodeDecode"))
do_statistics(extract_data(abab, "org.jetbrains.stdlibprofiling.TwitterBenchmark.encodeDecode"))


4431.810 (0.020)   
4386.150 (0.025)   


It seems this doesn't have effect. Which is good on one hand (because that would be quite hard to mitigate). On the
other hand, the source of problems is still unknown.

I also considered measuring the effect of `-Xmx` and `-Xms`. However, it's possible these were ignored earlier because
it was passed incorrectly -- `java -jar program.jar -Xmx48g`. Or maybe it was caused by me imprecise history keeping and
they were originally in the correct place.